# Multitemporal Surface Water Monitoring of Poyang Lake with Sentinel-2

This notebook implements a reproducible, cloud-based pipeline for mapping the
seasonal open-water extent of Poyang Lake (Jiangxi, China) from Sentinel-2
Surface Reflectance imagery over 2016–2020. For each year it builds wet- and
dry-season median composites in Google Earth Engine and extracts water with
three methods — NDWI (Otsu) thresholding, Random Forest and Support Vector
Machine — which are compared through accuracy assessment and a water-area time
series (seasonal and interannual change detection).

The notebook is organised into numbered sections. All tunable parameters
(area of interest, years, season months, spectral indices, classifier
hyperparameters) are centralised in the `CONFIG` block in Section 1.
It requires a Google Cloud project with the Earth Engine API enabled
(non-commercial / academic use), whose id must be set in the initialisation
cell below.


## 0. Setup

In [ ]:
!pip install -q earthengine-api geemap geopandas


In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Authenticate and initialise Earth Engine.
# Replace the project id below with a Google Cloud project that has the
# Earth Engine API enabled.
ee.Authenticate()
ee.Initialize(project="your-gee-project-id")


## 1. Configuration

The values below are parameters rather than fixed choices; they correspond to the methodological decisions justified in the article (Section "Description of the used EO tools").


In [ ]:
CONFIG = {
    # --- Study area: Poyang Lake, Jiangxi Province, China ---
    # Bounding box around the lake (lon_min, lat_min, lon_max, lat_max).
    # Tightened to the lake body + floodplain (drops surrounding land that
    # only adds computation). Widen again if a wet-season composite looks clipped.
    "aoi_bbox": [115.75, 28.40, 116.75, 29.55],

    # --- Sentinel-2 collection ---
    "s2_collection": "COPERNICUS/S2_SR_HARMONIZED",
    "cloud_prob_collection": "COPERNICUS/S2_CLOUD_PROBABILITY",
    "max_cloud_prob": 40,          # used to mask individual images before compositing
    "scale": 30,                   # working resolution in meters. 30 m matches the
                                    # JRC reference data and keeps free-tier memory in check
                                    # (20 m is feasible on a paid / high-memory Earth Engine project)

    # --- Study period ---
    # The JRC Global Surface Water YEARLY reference used for training/validation
    # only extends through 2021, so every study year must be <= 2021 (otherwise
    # the reference image is empty and the whole composite fails). We therefore
    # use 5 evenly-spaced years within 2016-2021. This still satisfies the
    # "minimum two epochs, more is better" requirement.
    "years": [2016, 2017, 2018, 2019, 2020],

    # --- Hydrological seasons for Poyang Lake ---
    # Wet / flood season: water expands, roughly April-September
    # Dry season: water retreats to the river channels, roughly November-March
    # These are a starting point from the general literature on Poyang Lake
    # hydrology -- refine if inspection of the composites
    # (Section 4) suggests the transition months should shift.
    "wet_season_months": (4, 9),        # inclusive, (start_month, end_month)
    "dry_season_months": (11, 3),       # wraps around the year end (Nov..Mar)

    # --- Reference / validation data ---
    "jrc_yearly": "JRC/GSW1_4/YearlyHistory",   # per-year water/no-water reference
    "jrc_occurrence": "JRC/GSW1_4/GlobalSurfaceWater",

    # --- Water extraction: Method 1, NDWI thresholding ---
    "ndwi_bands": {"green": "B3", "nir": "B8"},

    # --- Classification: Methods 2 & 3 (RF, SVM) ---
    # Use the SCALED reflectance bands (0-1) plus the indices, so SVM and RF
    # both see features on a comparable numeric range.
    "classification_bands": ["B2_s", "B3_s", "B4_s", "B8_s", "B11_s", "B12_s",
                              "NDWI", "MNDWI", "NDVI"],
    "n_training_points": 2000,     # stratified random points per composite
    "train_fraction": 0.7,         # rest goes to the held-out accuracy-assessment set
    "rf_trees": 100,
    "svm_kernel": "RBF",
    "svm_gamma": 0.5,
    "svm_cost": 10,

    "random_seed": 42,
}

aoi = ee.Geometry.Rectangle(CONFIG["aoi_bbox"])
print("AOI area (km^2):", aoi.area().divide(1e6).getInfo())


## 2. Study area preview

The AOI is displayed on an interactive map together with the JRC permanent/seasonal water layer, to confirm that the bounding box contains the whole lake and its floodplain.


In [ ]:
gsw_occurrence = ee.Image(CONFIG["jrc_occurrence"]).select("occurrence").clip(aoi)

Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(aoi, {"color": "red"}, "AOI (bounding box)")
Map.addLayer(gsw_occurrence.updateMask(gsw_occurrence.gt(0)),
             {"min": 0, "max": 100, "palette": ["ffffff", "0000ff"]},
             "JRC water occurrence 1984-2021 (%)")
Map


## 3. Cloud masking and spectral indices

Sentinel-2 SR (`COPERNICUS/S2_SR_HARMONIZED`) is masked using the companion `s2cloudless` cloud-probability collection rather than the coarser QA60 band, which gives cleaner composites over a large, frequently cloudy sub-tropical region like Jiangxi.

`NDWI`, `MNDWI` and `NDVI` are computed on every image so they are available both for thresholding (Method 1) and as extra classifier features (Methods 2-3).


In [ ]:
def mask_s2_clouds(image):
    """Attach cloud probability and mask out probable clouds."""
    cloud_prob = ee.Image(image.get("cloud_mask")).select("probability")
    is_cloud = cloud_prob.gte(CONFIG["max_cloud_prob"])
    return image.updateMask(is_cloud.Not()).copyProperties(image, ["system:time_start"])


def add_indices(image):
    ndwi = image.normalizedDifference(
        [CONFIG["ndwi_bands"]["green"], CONFIG["ndwi_bands"]["nir"]]
    ).rename("NDWI")
    mndwi = image.normalizedDifference(["B3", "B11"]).rename("MNDWI")
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    # Scaled reflectance copies (0-1 range) for classification. SVM is very
    # sensitive to feature scale: raw reflectance (0-3000+) swamps the -1..1
    # indices and makes the SVM collapse to a single class. Dividing by 10000
    # puts every optical band on the same 0-1 footing as the indices.
    scaled = image.select(["B2", "B3", "B4", "B8", "B11", "B12"]).divide(10000) \
        .rename(["B2_s", "B3_s", "B4_s", "B8_s", "B11_s", "B12_s"])
    return image.addBands([ndwi, mndwi, ndvi, scaled])


def get_s2_collection(start, end, geometry):
    s2 = ee.ImageCollection(CONFIG["s2_collection"]).filterDate(start, end).filterBounds(geometry)
    clouds = ee.ImageCollection(CONFIG["cloud_prob_collection"]).filterDate(start, end).filterBounds(geometry)

    joined = ee.Join.saveFirst("cloud_mask").apply(
        primary=s2,
        secondary=clouds,
        condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
    )
    s2_masked = ee.ImageCollection(joined).map(mask_s2_clouds).map(add_indices)
    return s2_masked


## 4. Seasonal composites (wet / dry season, per year)

For every year in `CONFIG["years"]` we build one **wet-season** and one **dry-season** median composite. Median compositing is a simple, robust way to suppress residual clouds/shadows and speckle-like noise without needing a per-pixel quality-scoring algorithm.


In [ ]:
def season_date_ranges(year, months):
    """Return (start, end) ee.Date strings for a season, handling wrap-around
    seasons such as Nov-Mar (dry season spans the turn of the year)."""
    start_month, end_month = months
    if start_month <= end_month:
        start = f"{year}-{start_month:02d}-01"
        end_year, end_month_ = year, end_month
    else:
        # season starts in `year` and ends in `year + 1`
        start = f"{year}-{start_month:02d}-01"
        end_year, end_month_ = year + 1, end_month
    if end_month_ == 12:
        end = f"{end_year}-12-31"
    else:
        end = f"{end_year}-{end_month_ + 1:02d}-01"
    return start, end


def build_seasonal_composite(year, season):
    months = CONFIG["wet_season_months"] if season == "wet" else CONFIG["dry_season_months"]
    start, end = season_date_ranges(year, months)
    coll = get_s2_collection(start, end, aoi)
    n_images = coll.size()
    composite = coll.median().clip(aoi)
    return composite.set({"year": year, "season": season, "start": start,
                           "end": end, "n_images": n_images})


# Build every composite once and cache it in a dict: composites[(year, "wet")], composites[(year, "dry")]
composites = {}
for y in CONFIG["years"]:
    composites[(y, "wet")] = build_seasonal_composite(y, "wet")
    composites[(y, "dry")] = build_seasonal_composite(y, "dry")

print(f"Built {len(composites)} seasonal composites for {len(CONFIG['years'])} years.")

# Diagnostic: how many usable (cloud-masked) images went into each composite?
# Season windows with very few images (e.g. 0-2) produce holey/unreliable
# composites -- these are the ones likely to fail or give odd results.
print("\nImage counts per composite (watch for low numbers):")
for (yy, ss), img in sorted(composites.items()):
    try:
        n = img.get("n_images").getInfo()
        flag = "  <-- LOW" if n is not None and n < 3 else ""
        print(f"  {yy} {ss}: {n} images{flag}")
    except Exception as e:
        print(f"  {yy} {ss}: could not count ({str(e)[:40]})")


## 5. Visual check of a couple of composites

A wet-season and a dry-season composite for the same year are compared in true colour and in NDWI, as a visual check before further processing.


In [ ]:
check_year = CONFIG["years"][-1]  # most recent year by default
wet_img = composites[(check_year, "wet")]
dry_img = composites[(check_year, "dry")]

vis_rgb = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
vis_ndwi = {"bands": ["NDWI"], "min": -0.5, "max": 0.5,
            "palette": ["8B4513", "ffffff", "0000ff"]}

Map2 = geemap.Map()
Map2.centerObject(aoi, 9)
Map2.addLayer(wet_img, vis_rgb, f"{check_year} wet season RGB")
Map2.addLayer(dry_img, vis_rgb, f"{check_year} dry season RGB")
Map2.addLayer(wet_img.select("NDWI"), vis_ndwi, f"{check_year} wet season NDWI")
Map2.addLayer(dry_img.select("NDWI"), vis_ndwi, f"{check_year} dry season NDWI")
Map2


## 6. Method 1 — NDWI thresholding (Otsu)

The draft proposes McFeeters' NDWI = (Green − NIR) / (Green + NIR). Rather than a single fixed threshold picked by eye (which does not generalize well across five years and two very different seasons), we compute an **Otsu threshold per composite**: NDWI histogram values are binned, and the threshold that best separates two classes (maximizing between-class variance) is chosen automatically. A fixed fallback threshold (0.0) is used if Otsu fails (e.g. too few water pixels in a very dry composite).


In [ ]:
def otsu_threshold(histogram):
    """Classic Otsu threshold on an ee histogram dictionary.
    Returns the NDWI value (an ee.Number) that maximizes between-class variance.
    Implemented by building a FeatureCollection of (threshold, bss) pairs and
    sorting -- robust and avoids ee.Array.argmax() index-unwrapping issues."""
    counts = ee.Array(ee.Dictionary(histogram).get("histogram"))
    means = ee.Array(ee.Dictionary(histogram).get("bucketMeans"))
    size = means.length().get([0])
    total = counts.reduce(ee.Reducer.sum(), [0]).get([0])
    sums = means.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0])
    mean = sums.divide(total)

    indices = ee.List.sequence(1, size.subtract(1))

    def make_pair(i):
        i = ee.Number(i)
        a_counts = counts.slice(0, 0, i)
        a_count = a_counts.reduce(ee.Reducer.sum(), [0]).get([0])
        a_means = means.slice(0, 0, i)
        a_mean = a_means.multiply(a_counts).reduce(ee.Reducer.sum(), [0]).get([0]).divide(a_count)
        b_count = total.subtract(a_count)
        b_mean = sums.subtract(a_count.multiply(a_mean)).divide(b_count)
        bss = a_count.multiply(a_mean.subtract(mean).pow(2)).add(
            b_count.multiply(b_mean.subtract(mean).pow(2)))
        return ee.Feature(None, {"bss": bss, "threshold": means.get([i])})

    pairs = ee.FeatureCollection(indices.map(make_pair))
    # the feature with the largest between-class variance = best threshold
    best = pairs.sort("bss", False).first()
    return ee.Number(best.get("threshold"))


def ndwi_water_mask(image, geometry, fallback=0.0):
    hist = image.select("NDWI").reduceRegion(
        reducer=ee.Reducer.histogram(255, 0.01),
        geometry=geometry, scale=CONFIG["scale"], maxPixels=1e13,
        bestEffort=True, tileScale=16,
    ).get("NDWI")

    threshold = ee.Algorithms.If(hist, otsu_threshold(hist), fallback)
    threshold = ee.Number(threshold)
    water = image.select("NDWI").gt(threshold).rename("water_ndwi")
    return water.set("ndwi_threshold", threshold)


ndwi_water = {}
for key, img in composites.items():
    ndwi_water[key] = ndwi_water_mask(img, aoi)

# Sanity check on one composite
sample_thr = ndwi_water[(check_year, "wet")].get("ndwi_threshold").getInfo()
print(f"Otsu NDWI threshold, {check_year} wet season: {sample_thr:.3f}")


## 7. Reference / training labels (JRC Global Surface Water)

There is no manual field survey available, so labels for both **training** the supervised classifiers and **validating** all three methods come from the JRC Global Surface Water Yearly History product (`JRC/GSW1_4/YearlyHistory`), an independent Landsat-derived dataset. Its classes are remapped to a binary Water / Not-water mask:

| JRC value | Meaning        | Remapped |
|-----------|----------------|----------|
| 0         | no observation | masked out |
| 1         | not water      | 0 |
| 2         | seasonal water | **0** (dry mudflat for most of the year) |
| 3         | permanent water| 1 |

**Why permanent water only?** The JRC yearly product flags a pixel as "seasonal water" if it held water at *any* time that year. In a dry-season composite those pixels are exposed mudflat, yet a seasonal→water label would train the classifier to call mudflat "water", which badly over-predicts water in the dry season (visually confirmed on the composites). Restricting the positive class to permanent water gives a stable, season-consistent target. Because JRC is a *yearly* (not seasonal) product it is still used as a same-year reference for both seasons — a simplification acknowledged in the limitations.


In [ ]:
def jrc_reference_for_year(year):
    coll = ee.ImageCollection(CONFIG["jrc_yearly"])
    img = coll.filter(ee.Filter.calendarRange(year, year, "year")).first()
    # PERMANENT water only (class 3) counts as water. Seasonal water (class 2)
    # is treated as non-water, because in the dry-season composite those pixels
    # are exposed mudflat -- labelling them "water" made the classifier learn to
    # call dry mudflat "water" and massively over-predicted water in the dry season.
    # Using permanent water gives a stable, season-consistent training target.
    reference = img.select("waterClass").remap([0, 1, 2, 3], [0, 0, 0, 1], 0).rename("water_ref")
    return reference.updateMask(img.select("waterClass").neq(0)).clip(aoi)


reference_by_year = {y: jrc_reference_for_year(y) for y in CONFIG["years"]}
print("Reference layers ready for years:", list(reference_by_year.keys()))


## 8. Methods 2 & 3 — Random Forest and SVM classification

For every seasonal composite:
1. Stack the classification bands (`CONFIG["classification_bands"]`).
2. Draw `CONFIG["n_training_points"]` stratified random points labelled from the JRC reference of the same year.
3. Split points into train / test (`CONFIG["train_fraction"]`).
4. Train an `ee.Classifier.smileRandomForest` and an `ee.Classifier.libsvm` on the training points.
5. Classify the whole composite with both.

This is wrapped in one function so it is trivial to loop over every (year, season) pair afterwards.


In [ ]:
def sample_points(image, reference, geometry, n_points):
    stack = image.select(CONFIG["classification_bands"]).addBands(reference)
    samples = stack.stratifiedSample(
        numPoints=n_points,
        classBand="water_ref",
        region=geometry,
        scale=CONFIG["scale"],
        seed=CONFIG["random_seed"],
        geometries=True,
        tileScale=16,   # split the sampling into smaller tiles -> stays under
                        # the free-tier memory limit ("User memory limit exceeded")
    )
    samples = samples.randomColumn("rand", CONFIG["random_seed"])
    train = samples.filter(ee.Filter.lt("rand", CONFIG["train_fraction"]))
    test = samples.filter(ee.Filter.gte("rand", CONFIG["train_fraction"]))
    return train, test


def classify_rf_svm(image, geometry, reference, n_points=None):
    n_points = n_points or CONFIG["n_training_points"]
    train, test = sample_points(image, reference, geometry, n_points)
    bands = CONFIG["classification_bands"]

    rf = ee.Classifier.smileRandomForest(CONFIG["rf_trees"]).train(
        features=train, classProperty="water_ref", inputProperties=bands
    )
    svm = ee.Classifier.libsvm(
        kernelType=CONFIG["svm_kernel"], gamma=CONFIG["svm_gamma"], cost=CONFIG["svm_cost"]
    ).train(features=train, classProperty="water_ref", inputProperties=bands)

    rf_class = image.select(bands).classify(rf).rename("water_rf")
    svm_class = image.select(bands).classify(svm).rename("water_svm")

    return {
        "rf_image": rf_class,
        "svm_image": svm_class,
        "rf_classifier": rf,
        "svm_classifier": svm,
        "train": train,
        "test": test,
    }


# Run for every (year, season): this is the expensive step. Each call submits
# server-side work; results below are only computed lazily, so this cell
# itself is fast -- the actual computation is triggered later by getInfo()/
# reduceRegion()/Export in Sections 9-10.
classification_results = {}
for (y, season), img in composites.items():
    reference = reference_by_year[y]
    classification_results[(y, season)] = classify_rf_svm(img, aoi, reference)

print("Classifiers built for", len(classification_results), "composites (RF + SVM each).")


## 9. Accuracy assessment

For every (year, season) and every method (NDWI-Otsu, RF, SVM) compute a confusion matrix against the held-out test points (RF/SVM) or against a fresh sample of JRC reference points (NDWI), then derive Overall Accuracy, Precision, Recall, F1-score and Cohen's kappa. Results are collected into one tidy `pandas` DataFrame for easy comparison / plotting / inclusion as a table in the article.


In [ ]:
def confusion_metrics(confusion_array):
    """confusion_array: 2x2 nested list, rows=truth, cols=predicted, class order [0,1]"""
    cm = np.array(confusion_array)
    tn, fp = cm[0, 0], cm[0, 1]
    fn, tp = cm[1, 0], cm[1, 1]
    total = tn + fp + fn + tp
    oa = (tp + tn) / total if total else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision and recall else np.nan
    po = oa
    pe = (((tp + fp) * (tp + fn)) + ((fn + tn) * (fp + tn))) / total ** 2 if total else np.nan
    kappa = (po - pe) / (1 - pe) if pe != 1 else np.nan
    return {"overall_accuracy": oa, "precision": precision, "recall": recall,
            "f1_score": f1, "kappa": kappa}


def assess_rf_svm(result):
    out = {}
    for method in ["rf", "svm"]:
        classifier = result[f"{method}_classifier"]
        test = result["test"]
        classified_test = test.classify(classifier)
        cm = classified_test.errorMatrix("water_ref", "classification").array().getInfo()
        out[method] = confusion_metrics(cm)
    return out


def assess_ndwi_at_points(ndwi_mask_image, test_points):
    """Evaluate the NDWI water mask at the SAME held-out test points used for
    RF/SVM, instead of drawing a fresh full-image sample. Much lighter on memory
    and makes the three methods directly comparable on identical points."""
    sampled = ndwi_mask_image.rename("classification").sampleRegions(
        collection=test_points,
        scale=CONFIG["scale"],
        geometries=False,
    )
    cm = sampled.errorMatrix("water_ref", "classification").array().getInfo()
    return confusion_metrics(cm)


import time

def retry(fn, tries=3, wait=15):
    """Retry a server call a few times: free-tier GEE sometimes throws
    transient 'memory limit exceeded' errors that succeed on a second try."""
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            if attempt == tries - 1:
                raise
            print(f"    retry {attempt + 1}/{tries - 1} after error: {str(e)[:60]}...")
            time.sleep(wait)


# Process ONE (year, season) at a time so we never hold too much on the
# server at once. This is slower but stays within the free-tier memory limit.
rows = []
failed = []
for (y, season), result in classification_results.items():
    print(f"Assessing {y} {season} ...")
    reference = reference_by_year[y]
    try:
        rf_svm_metrics = retry(lambda: assess_rf_svm(result))
        ndwi_metrics = retry(lambda: assess_ndwi_at_points(ndwi_water[(y, season)], result["test"]))
    except Exception as e:
        print(f"    SKIPPED {y} {season}: {str(e)[:70]}")
        failed.append((y, season, str(e)[:70]))
        continue

    for method, metrics in [("NDWI_Otsu", ndwi_metrics),
                             ("RandomForest", rf_svm_metrics["rf"]),
                             ("SVM", rf_svm_metrics["svm"])]:
        rows.append({"year": y, "season": season, "method": method, **metrics})

accuracy_df = pd.DataFrame(rows)
accuracy_df.to_csv("accuracy_assessment.csv", index=False)
print(f"\nDone. {len(rows)//3} composites assessed, {len(failed)} skipped.")
if failed:
    print("Skipped (inspect these composites manually - likely too few clear images):")
    for f in failed:
        print("   ", f[0], f[1])
accuracy_df.sort_values(["year", "season", "method"]).head(20)


## 10. Comparing the three methods

Aggregate the per-(year, season) metrics into a single table per method — this is the table to report as "Results" / method-comparison in the article.


In [ ]:
method_summary = accuracy_df.groupby("method")[
    ["overall_accuracy", "precision", "recall", "f1_score", "kappa"]
].mean().round(3)
print(method_summary)

fig, ax = plt.subplots(figsize=(7, 4))
for method in accuracy_df["method"].unique():
    sub = accuracy_df[accuracy_df["method"] == method].sort_values("year")
    ax.plot(sub["year"].astype(str) + "-" + sub["season"], sub["overall_accuracy"],
            marker="o", label=method)
ax.set_ylabel("Overall accuracy")
ax.set_xlabel("Year - season")
ax.set_title("Overall accuracy by method, year and season")
ax.legend()
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=200)
plt.show()


## 11. Water extent time series & change detection

For every (year, season, method) compute the water surface area in km² by summing pixel area over the water mask. This gives:
- the **multi-year time series** (2016--2020) for wet and dry seasons separately, i.e. interannual variability;
- the **wet-minus-dry difference per year** → flood-season expansion;
- one time series per method, showing whether NDWI/RF/SVM agree on the trend or diverge.


In [ ]:
def water_area_km2(water_mask_image, geometry, band_name):
    area_img = water_mask_image.select(band_name).multiply(ee.Image.pixelArea())
    stats = area_img.reduceRegion(
        reducer=ee.Reducer.sum(), geometry=geometry,
        scale=60, maxPixels=1e13, bestEffort=True, tileScale=16,
    )
    return ee.Number(stats.get(band_name)).divide(1e6)  # m^2 -> km^2


area_rows = []
for (y, season), result in classification_results.items():
    print(f"Measuring water area {y} {season} ...")
    try:
        ndwi_area = retry(lambda: water_area_km2(ndwi_water[(y, season)], aoi, "water_ndwi").getInfo())
        rf_area = retry(lambda: water_area_km2(result["rf_image"], aoi, "water_rf").getInfo())
        svm_area = retry(lambda: water_area_km2(result["svm_image"], aoi, "water_svm").getInfo())
    except Exception as e:
        print(f"    SKIPPED {y} {season}: {str(e)[:70]}")
        continue
    area_rows.append({"year": y, "season": season,
                       "NDWI_Otsu_km2": ndwi_area,
                       "RandomForest_km2": rf_area,
                       "SVM_km2": svm_area})

area_df = pd.DataFrame(area_rows).sort_values(["year", "season"]).reset_index(drop=True)
area_df.to_csv("water_area_timeseries.csv", index=False)
print(f"\nDone. {len(area_rows)} composites measured.")

# --- Flag cloud-corrupted composites (data-availability outliers) ---
# The 2018 wet-season median composite is built from too few clear Sentinel-2
# scenes (heavy Apr-Sep cloud cover that year), so its water area collapses to
# an implausible ~80-400 km2 -- far below even the same year's dry season. It is
# a data-availability artefact, NOT a real hydrological signal, so it is excluded
# from the trend plots and seasonal-expansion analysis (kept in the raw CSV for
# full transparency, and discussed explicitly in the article's Discussion).
EXCLUDE = [(2018, "wet")]

def is_excluded(row):
    return (row["year"], row["season"]) in EXCLUDE

area_df_clean = area_df[~area_df.apply(is_excluded, axis=1)].reset_index(drop=True)
area_df_clean.to_csv("water_area_timeseries_clean.csv", index=False)
print("Excluded from trend analysis (cloud-corrupted composites):", EXCLUDE)
area_df


In [ ]:
# Multi-year time series plot: wet vs dry season, one line per method
# Uses area_df_clean (2018 wet excluded as a cloud-corrupted outlier).
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, season in zip(axes, ["wet", "dry"]):
    sub = area_df_clean[area_df_clean["season"] == season]
    for col, label in [("NDWI_Otsu_km2", "NDWI (Otsu)"),
                        ("RandomForest_km2", "Random Forest"),
                        ("SVM_km2", "SVM")]:
        ax.plot(sub["year"], sub[col], marker="o", label=label)
    ax.set_title(f"{season.capitalize()} season water extent")
    ax.set_xlabel("Year")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Water area (km²)")
axes[0].legend()
plt.tight_layout()
plt.savefig("water_extent_timeseries.png", dpi=200)
plt.show()


In [ ]:
# Seasonal expansion (flood season minus dry season, per year) and a simple
# linear trend per season to quantify interannual variability.
# Built from area_df_clean, so 2018 (whose wet composite was excluded) drops out
# of the expansion analysis rather than contributing a spurious negative value.
pivot = area_df_clean.pivot(index="year", columns="season",
                       values=["NDWI_Otsu_km2", "RandomForest_km2", "SVM_km2"])

expansion = pd.DataFrame({
    method: pivot[method]["wet"] - pivot[method]["dry"]
    for method in ["NDWI_Otsu_km2", "RandomForest_km2", "SVM_km2"]
}).dropna()
expansion.index.name = "year"
expansion.to_csv("seasonal_expansion.csv")
print("Seasonal (wet - dry) expansion, km^2  (2018 excluded: no valid wet composite):")
print(expansion.round(1))

for method, label in [("NDWI_Otsu_km2", "NDWI (Otsu)"),
                      ("RandomForest_km2", "Random Forest")]:
    years = expansion.index.values.astype(float)
    slope, intercept = np.polyfit(years, expansion[method].values, 1)
    print(f"\n{label}: interannual trend in seasonal expansion "
          f"= {slope:.2f} km^2/year")


## 12. Maps for the article: multitemporal & seasonal comparison

Export a small multi-panel comparison for a chosen year (wet vs dry, and the three methods) as static images, plus optionally push full-resolution GeoTIFFs to Google Drive for use as figures in the paper / QGIS.


In [ ]:
figure_year = CONFIG["years"][-1]  # year used for the example maps

Map3 = geemap.Map()
Map3.centerObject(aoi, 9)
for season in ["wet", "dry"]:
    r = classification_results[(figure_year, season)]
    Map3.addLayer(ndwi_water[(figure_year, season)].selfMask(),
                  {"palette": ["00BFFF"]}, f"{figure_year} {season} - NDWI water")
    Map3.addLayer(r["rf_image"].selfMask(),
                  {"palette": ["1f78b4"]}, f"{figure_year} {season} - RF water")
    Map3.addLayer(r["svm_image"].selfMask(),
                  {"palette": ["6a3d9a"]}, f"{figure_year} {season} - SVM water")
Map3


### Visual verification: RF water (red) over true-colour, wet vs dry

Static server-side thumbnails (do not need a browser screenshot engine, unlike
`geemap`'s `to_image`). Red = pixels classified as water by Random Forest, over
the true-colour composite. Use this to **critically inspect** whether the dry
season sensibly shows *less* water than the wet season, as expected for Poyang
Lake — exactly the kind of anomaly check the project guidelines ask for.


In [ ]:
from IPython.display import Image, display

preview_year = CONFIG["years"][-1]
vis_rgb = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}

for season in ["wet", "dry"]:
    r = classification_results[(preview_year, season)]
    comp = composites[(preview_year, season)]
    blended = comp.visualize(**vis_rgb).blend(
        r["rf_image"].selfMask().visualize(palette=["ff0000"])
    )
    url = blended.getThumbURL({"region": aoi, "dimensions": 700, "format": "png"})
    print(f"===== {preview_year} {season}: RF water (RED) over true-colour =====")
    display(Image(url=url))


In [ ]:
# Optional: export a chosen classified image to Google Drive as a GeoTIFF
# (uncomment to run when the file is needed; each export is a
# background Earth Engine task, check progress at code.earthengine.google.com/tasks)

# export_image = classification_results[(figure_year, "wet")]["rf_image"]
# task = ee.batch.Export.image.toDrive(
#     image=export_image,
#     description=f"PoyangLake_RF_{figure_year}_wet",
#     folder="EO_Advanced_Poyang",
#     region=aoi,
#     scale=CONFIG["scale"],
#     maxPixels=1e13,
# )
# task.start()


## 13. Summary outputs for the article

Files written into the working directory by this notebook (they can be downloaded, or Drive can be mounted first for persistence):

- `accuracy_assessment.csv` — per year/season/method confusion-matrix-derived metrics
- `accuracy_comparison.png` — accuracy comparison figure
- `water_area_timeseries.csv` — water extent (km²) time series, 2016-2020, wet & dry, 3 methods
- `water_extent_timeseries.png` — time series figure
- `seasonal_expansion.csv` — wet-minus-dry expansion per year, 3 methods

These four artifacts map directly onto the "Expected Outputs" in the project draft (seasonal composite maps, multitemporal water maps, water extent time-series, classification comparison results, accuracy tables) and are what Section 4 (Results) of the article should be built around.

**Known simplifications to acknowledge in the article's limitations:**
- Season boundaries (`wet_season_months` / `dry_season_months`) are literature-based defaults, not tuned per-year to the actual hydrograph.
- JRC yearly water history is used as reference/training for both seasonal composites of a year, even though it is not itself season-specific.
- Median compositing can still mix conditions within a 4-6 month window in years with unusually persistent cloud cover; inspect a few composites' image counts if a given year looks anomalous.
